In [73]:
import pandas as pd
from kessler.original.data import kelvins_to_event_dataset
import random

from kessler.io import CSVReader, CDM
from kessler.io.schemas import InputsSchema
import os
import numpy as np

In [74]:
column_mapping = {
    "CCSDS_CDM_VERS": "CCSDS_CDM_VERS",
    "CREATION_DATE": "CREATION_DATE",
    "ORIGINATOR": "ORIGINATOR",
    "MESSAGE_FOR": "MESSAGE_FOR",
    "MESSAGE_ID": "MESSAGE_ID",
    "TCA": "TCA",
    "MISS_DISTANCE": "MISS_DISTANCE",
    "RELATIVE_SPEED": "RELATIVE_SPEED",
    "RELATIVE_POSITION_R": "RELATIVE_POSITION_R",
    "RELATIVE_POSITION_T": "RELATIVE_POSITION_T",
    "RELATIVE_POSITION_N": "RELATIVE_POSITION_N",
    "RELATIVE_VELOCITY_R": "RELATIVE_VELOCITY_R",
    "RELATIVE_VELOCITY_T": "RELATIVE_VELOCITY_T",
    "RELATIVE_VELOCITY_N": "RELATIVE_VELOCITY_N",
    "START_SCREEN_PERIOD": "START_SCREEN_PERIOD",
    "STOP_SCREEN_PERIOD": "STOP_SCREEN_PERIOD",
    "SCREEN_VOLUME_FRAME": "SCREEN_VOLUME_FRAME",
    "SCREEN_VOLUME_SHAPE": "SCREEN_VOLUME_SHAPE",
    "SCREEN_VOLUME_X": "SCREEN_VOLUME_X",
    "SCREEN_VOLUME_Y": "SCREEN_VOLUME_Y",
    "SCREEN_VOLUME_Z": "SCREEN_VOLUME_Z",
    "SCREEN_ENTRY_TIME": "SCREEN_ENTRY_TIME",
    "SCREEN_EXIT_TIME": "SCREEN_EXIT_TIME",
    "COLLISION_PROBABILITY": "COLLISION_PROBABILITY",
    "COLLISION_PROBABILITY_METHOD": "COLLISION_PROBABILITY_METHOD",
    "OBJECT1_OBJECT": "t_OBJECT",
    "OBJECT1_OBJECT_DESIGNATOR": "t_OBJECT_DESIGNATOR",
    "OBJECT1_CATALOG_NAME": "t_CATALOG_NAME",
    "OBJECT1_OBJECT_NAME": "t_OBJECT_NAME",
    "OBJECT1_INTERNATIONAL_DESIGNATOR": "t_INTERNATIONAL_DESIGNATOR",
    "OBJECT1_OBJECT_TYPE": "t_OBJECT_TYPE",
    "OBJECT1_OPERATOR_CONTACT_POSITION": "t_OPERATOR_CONTACT_POSITION",
    "OBJECT1_OPERATOR_ORGANIZATION": "t_OPERATOR_ORGANIZATION",
    "OBJECT1_OPERATOR_PHONE": "t_OPERATOR_PHONE",
    "OBJECT1_OPERATOR_EMAIL": "t_OPERATOR_EMAIL",
    "OBJECT1_EPHEMERIS_NAME": "t_EPHEMERIS_NAME",
    "OBJECT1_COVARIANCE_METHOD": "t_COVARIANCE_METHOD",
    "OBJECT1_MANEUVERABLE": "t_MANEUVERABLE",
    "OBJECT1_ORBIT_CENTER": "t_ORBIT_CENTER",
    "OBJECT1_REF_FRAME": "t_REF_FRAME",
    "OBJECT1_GRAVITY_MODEL": "t_GRAVITY_MODEL",
    "OBJECT1_ATMOSPHERIC_MODEL": "t_ATMOSPHERIC_MODEL",
    "OBJECT1_N_BODY_PERTURBATIONS": "t_N_BODY_PERTURBATIONS",
    "OBJECT1_SOLAR_RAD_PRESSURE": "t_SOLAR_RAD_PRESSURE",
    "OBJECT1_EARTH_TIDES": "t_EARTH_TIDES",
    "OBJECT1_INTRACK_THRUST": "t_INTRACK_THRUST",
    "OBJECT1_TIME_LASTOB_START": "t_TIME_LASTOB_START",
    "OBJECT1_TIME_LASTOB_END": "t_TIME_LASTOB_END",
    "OBJECT1_RECOMMENDED_OD_SPAN": "t_RECOMMENDED_OD_SPAN",
    "OBJECT1_ACTUAL_OD_SPAN": "t_ACTUAL_OD_SPAN",
    "OBJECT1_OBS_AVAILABLE": "t_OBS_AVAILABLE",
    "OBJECT1_OBS_USED": "t_OBS_USED",
    "OBJECT1_TRACKS_AVAILABLE": "t_TRACKS_AVAILABLE",
    "OBJECT1_TRACKS_USED": "t_TRACKS_USED",
    "OBJECT1_RESIDUALS_ACCEPTED": "t_RESIDUALS_ACCEPTED",
    "OBJECT1_WEIGHTED_RMS": "t_WEIGHTED_RMS",
    "OBJECT1_AREA_PC": "t_AREA_PC",
    "OBJECT1_AREA_DRG": "t_AREA_DRG",
    "OBJECT1_AREA_SRP": "t_AREA_SRP",
    "OBJECT1_MASS": "t_MASS",
    "OBJECT1_CD_AREA_OVER_MASS": "t_CD_AREA_OVER_MASS",
    "OBJECT1_CR_AREA_OVER_MASS": "t_CR_AREA_OVER_MASS",
    "OBJECT1_THRUST_ACCELERATION": "t_THRUST_ACCELERATION",
    "OBJECT1_SEDR": "t_SEDR",
    "OBJECT1_X": "t_X",
    "OBJECT1_Y": "t_Y",
    "OBJECT1_Z": "t_Z",
    "OBJECT1_X_DOT": "t_X_DOT",
    "OBJECT1_Y_DOT": "t_Y_DOT",
    "OBJECT1_Z_DOT": "t_Z_DOT",
    "OBJECT1_CR_R": "t_CR_R",
    "OBJECT1_CT_R": "t_CT_R",
    "OBJECT1_CT_T": "t_CT_T",
    "OBJECT1_CN_R": "t_CN_R",
    "OBJECT1_CN_T": "t_CN_T",
    "OBJECT1_CN_N": "t_CN_N",
    "OBJECT1_CRDOT_R": "t_CRDOT_R",
    "OBJECT1_CRDOT_T": "t_CRDOT_T",
    "OBJECT1_CRDOT_N": "t_CRDOT_N",
    "OBJECT1_CRDOT_RDOT": "t_CRDOT_RDOT",
    "OBJECT1_CTDOT_R": "t_CTDOT_R",
    "OBJECT1_CTDOT_T": "t_CTDOT_T",
    "OBJECT1_CTDOT_N": "t_CTDOT_N",
    "OBJECT1_CTDOT_RDOT": "t_CTDOT_RDOT",
    "OBJECT1_CTDOT_TDOT": "t_CTDOT_TDOT",
    "OBJECT1_CNDOT_R": "t_CNDOT_R",
    "OBJECT1_CNDOT_T": "t_CNDOT_T",
    "OBJECT1_CNDOT_N": "t_CNDOT_N",
    "OBJECT1_CNDOT_RDOT": "t_CNDOT_RDOT",
    "OBJECT1_CNDOT_TDOT": "t_CNDOT_TDOT",
    "OBJECT1_CNDOT_NDOT": "t_CNDOT_NDOT",
    "OBJECT1_CDRG_R": "t_CDRG_R",
    "OBJECT1_CDRG_T": "t_CDRG_T",
    "OBJECT1_CDRG_N": "t_CDRG_N",
    "OBJECT1_CDRG_RDOT": "t_CDRG_RDOT",
    "OBJECT1_CDRG_TDOT": "t_CDRG_TDOT",
    "OBJECT1_CDRG_NDOT": "t_CDRG_NDOT",
    "OBJECT1_CDRG_DRG": "t_CDRG_DRG",
    "OBJECT1_CSRP_R": "t_CSRP_R",
    "OBJECT1_CSRP_T": "t_CSRP_T",
    "OBJECT1_CSRP_N": "t_CSRP_N",
    "OBJECT1_CSRP_RDOT": "t_CSRP_RDOT",
    "OBJECT1_CSRP_TDOT": "t_CSRP_TDOT",
    "OBJECT1_CSRP_NDOT": "t_CSRP_NDOT",
    "OBJECT1_CSRP_DRG": "t_CSRP_DRG",
    "OBJECT1_CSRP_SRP": "t_CSRP_SRP",
    "OBJECT1_CTHR_R": "t_CTHR_R",
    "OBJECT1_CTHR_T": "t_CTHR_T",
    "OBJECT1_CTHR_N": "t_CTHR_N",
    "OBJECT1_CTHR_RDOT": "t_CTHR_RDOT",
    "OBJECT1_CTHR_TDOT": "t_CTHR_TDOT",
    "OBJECT1_CTHR_NDOT": "t_CTHR_NDOT",
    "OBJECT1_CTHR_DRG": "t_CTHR_DRG",
    "OBJECT1_CTHR_SRP": "t_CTHR_SRP",
    "OBJECT1_CTHR_THR": "t_CTHR_THR",
    "OBJECT2_OBJECT": "c_OBJECT",
    "OBJECT2_OBJECT_DESIGNATOR": "c_OBJECT_DESIGNATOR",
    "OBJECT2_CATALOG_NAME": "c_CATALOG_NAME",
    "OBJECT2_OBJECT_NAME": "c_OBJECT_NAME",
    "OBJECT2_INTERNATIONAL_DESIGNATOR": "c_INTERNATIONAL_DESIGNATOR",
    "OBJECT2_OBJECT_TYPE": "c_OBJECT_TYPE",
    "OBJECT2_OPERATOR_CONTACT_POSITION": "c_OPERATOR_CONTACT_POSITION",
    "OBJECT2_OPERATOR_ORGANIZATION": "c_OPERATOR_ORGANIZATION",
    "OBJECT2_OPERATOR_PHONE": "c_OPERATOR_PHONE",
    "OBJECT2_OPERATOR_EMAIL": "c_OPERATOR_EMAIL",
    "OBJECT2_EPHEMERIS_NAME": "c_EPHEMERIS_NAME",
    "OBJECT2_COVARIANCE_METHOD": "c_COVARIANCE_METHOD",
    "OBJECT2_MANEUVERABLE": "c_MANEUVERABLE",
    "OBJECT2_ORBIT_CENTER": "c_ORBIT_CENTER",
    "OBJECT2_REF_FRAME": "c_REF_FRAME",
    "OBJECT2_GRAVITY_MODEL": "c_GRAVITY_MODEL",
    "OBJECT2_ATMOSPHERIC_MODEL": "c_ATMOSPHERIC_MODEL",
    "OBJECT2_N_BODY_PERTURBATIONS": "c_N_BODY_PERTURBATIONS",
    "OBJECT2_SOLAR_RAD_PRESSURE": "c_SOLAR_RAD_PRESSURE",
    "OBJECT2_EARTH_TIDES": "c_EARTH_TIDES",
    "OBJECT2_INTRACK_THRUST": "c_INTRACK_THRUST",
    "OBJECT2_TIME_LASTOB_START": "c_TIME_LASTOB_START",
    "OBJECT2_TIME_LASTOB_END": "c_TIME_LASTOB_END",
    "OBJECT2_RECOMMENDED_OD_SPAN": "c_RECOMMENDED_OD_SPAN",
    "OBJECT2_ACTUAL_OD_SPAN": "c_ACTUAL_OD_SPAN",
    "OBJECT2_OBS_AVAILABLE": "c_OBS_AVAILABLE",
    "OBJECT2_OBS_USED": "c_OBS_USED",
    "OBJECT2_TRACKS_AVAILABLE": "c_TRACKS_AVAILABLE",
    "OBJECT2_TRACKS_USED": "c_TRACKS_USED",
    "OBJECT2_RESIDUALS_ACCEPTED": "c_RESIDUALS_ACCEPTED",
    "OBJECT2_WEIGHTED_RMS": "c_WEIGHTED_RMS",
    "OBJECT2_AREA_PC": "c_AREA_PC",
    "OBJECT2_AREA_DRG": "c_AREA_DRG",
    "OBJECT2_AREA_SRP": "c_AREA_SRP",
    "OBJECT2_MASS": "c_MASS",
    "OBJECT2_CD_AREA_OVER_MASS": "c_CD_AREA_OVER_MASS",
    "OBJECT2_CR_AREA_OVER_MASS": "c_CR_AREA_OVER_MASS",
    "OBJECT2_THRUST_ACCELERATION": "c_THRUST_ACCELERATION",
    "OBJECT2_SEDR": "c_SEDR",
    "OBJECT2_X": "c_X",
    "OBJECT2_Y": "c_Y",
    "OBJECT2_Z": "c_Z",
    "OBJECT2_X_DOT": "c_X_DOT",
    "OBJECT2_Y_DOT": "c_Y_DOT",
    "OBJECT2_Z_DOT": "c_Z_DOT",
    "OBJECT2_CR_R": "c_CR_R",
    "OBJECT2_CT_R": "c_CT_R",
    "OBJECT2_CT_T": "c_CT_T",
    "OBJECT2_CN_R": "c_CN_R",
    "OBJECT2_CN_T": "c_CN_T",
    "OBJECT2_CN_N": "c_CN_N",
    "OBJECT2_CRDOT_R": "c_CRDOT_R",
    "OBJECT2_CRDOT_T": "c_CRDOT_T",
    "OBJECT2_CRDOT_N": "c_CRDOT_N",
    "OBJECT2_CRDOT_RDOT": "c_CRDOT_RDOT",
    "OBJECT2_CTDOT_R": "c_CTDOT_R",
    "OBJECT2_CTDOT_T": "c_CTDOT_T",
    "OBJECT2_CTDOT_N": "c_CTDOT_N",
    "OBJECT2_CTDOT_RDOT": "c_CTDOT_RDOT",
    "OBJECT2_CTDOT_TDOT": "c_CTDOT_TDOT",
    "OBJECT2_CNDOT_R": "c_CNDOT_R",
    "OBJECT2_CNDOT_T": "c_CNDOT_T",
    "OBJECT2_CNDOT_N": "c_CNDOT_N",
    "OBJECT2_CNDOT_RDOT": "c_CNDOT_RDOT",
    "OBJECT2_CNDOT_TDOT": "c_CNDOT_TDOT",
    "OBJECT2_CNDOT_NDOT": "c_CNDOT_NDOT",
    "OBJECT2_CDRG_R": "c_CDRG_R",
    "OBJECT2_CDRG_T": "c_CDRG_T",
    "OBJECT2_CDRG_N": "c_CDRG_N",
    "OBJECT2_CDRG_RDOT": "c_CDRG_RDOT",
    "OBJECT2_CDRG_TDOT": "c_CDRG_TDOT",
    "OBJECT2_CDRG_NDOT": "c_CDRG_NDOT",
    "OBJECT2_CDRG_DRG": "c_CDRG_DRG",
    "OBJECT2_CSRP_R": "c_CSRP_R",
    "OBJECT2_CSRP_T": "c_CSRP_T",
    "OBJECT2_CSRP_N": "c_CSRP_N",
    "OBJECT2_CSRP_RDOT": "c_CSRP_RDOT",
    "OBJECT2_CSRP_TDOT": "c_CSRP_TDOT",
    "OBJECT2_CSRP_NDOT": "c_CSRP_NDOT",
    "OBJECT2_CSRP_DRG": "c_CSRP_DRG",
    "OBJECT2_CSRP_SRP": "c_CSRP_SRP",
    "OBJECT2_CTHR_R": "c_CTHR_R",
    "OBJECT2_CTHR_T": "c_CTHR_T",
    "OBJECT2_CTHR_N": "c_CTHR_N",
    "OBJECT2_CTHR_RDOT": "c_CTHR_RDOT",
    "OBJECT2_CTHR_TDOT": "c_CTHR_TDOT",
    "OBJECT2_CTHR_NDOT": "c_CTHR_NDOT",
    "OBJECT2_CTHR_DRG": "c_CTHR_DRG",
    "OBJECT2_CTHR_SRP": "c_CTHR_SRP",
    "OBJECT2_CTHR_THR": "c_CTHR_THR",
    "__CREATION_DATE": "CREATION_DATE_IN_DAYS",
    "__TCA": "TCA_IN_DAYS",
    "__DAYS_TO_TCA": "DAYS_TO_TCA",
}

In [75]:
file_name = "data/train_data.csv"
number_of_events = 100

event_idx = 0
cdm_idx = 0


In [76]:
events_original = kelvins_to_event_dataset(
    file_name, drop_features=["c_rcs_estimate", "t_rcs_estimate"], num_events=100
)

cdm0_original = events_original._events[event_idx]._cdms[cdm_idx]
cdm0_orginal_df = cdm0_original.to_dataframe()
cdm0_orginal_df = cdm0_orginal_df.rename(columns=column_mapping)

Loading Kelvins dataset from file name: data/train_data.csv
162634 entries
Dropping features: ['c_rcs_estimate', 't_rcs_estimate']
Dropping rows with NaNs
146571 entries
Removing outliers
127037 entries
Shuffling
Grouped rows into 9586 events
Taking TCA as current time: 2025-06-18 20:10:22.438909
Converting Kelvins challenge data to EventDataset
Time spent  | Time remain.| Progress             | Events  | Events/sec
0d:00:00:00 | 0d:00:00:00 | #################### | 100/100 | 273.97       


In [77]:
cols_to_keep = dir(InputsSchema)
file_name = "data/train_data.csv"

events = CSVReader(path=file_name, number_of_events=100).read(columns_to_keep=cols_to_keep)
cdm0 = events.events[event_idx].cdms[cdm_idx]
cdm0_df = cdm0.to_dataframe()
cdm0_df = cdm0_df.drop(columns=["EVENT_ID"])

assert len(events) == number_of_events

2025-06-18 20:10:24.233 | WARNING  | kessler.io.csv:read:115 - The following columns are not in the dataframe: ['empty']
/Users/yogi/projects/kess/kessler/src/kessler/io/csv.py:186: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = data[condition]
/Users/yogi/projects/kess/kessler/src/kessler/io/csv.py:186: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = data[condition]
/Users/yogi/projects/kess/kessler/src/kessler/io/csv.py:186: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = data[condition]
/Users/yogi/projects/kess/kessler/src/kessler/io/csv.py:186: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = data[condition]
/Users/yogi/projects/kess/kessler/src/kessler/io/csv.py:186: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = data[condition]
2025-06-18 20:10:24.472 | WARNING  | kessler.io.csv:read:126 - Number

# Tests

In [78]:
# cdm copy

cdm0_copy = cdm0.copy()

assert cdm0_copy == cdm0

cdm1 = events.events[event_idx].cdms[cdm_idx + 1]
cdm0_copy.copy_from_other_cdm(cdm1)
assert cdm0_copy == cdm1

# use the right keys and values
# Create a dummy dict
assert len(cdm0.to_dict().keys()) == 209

# TODO: use the right column names
assert len(events.events[event_idx].cdms[cdm_idx].to_dataframe().columns) == 209

cdm0.save("cdm0.yml")

assert os.path.exists("cdm0.yml")

cdm0_loaded = CDM.load("cdm0.yml")

# TODO: dont know why this fails
# assert cdm0_loaded == cdm0

cdm0.set_header("EVENT_ID", 10)

assert cdm0.header.get("EVENT_ID") == 10


cdm0.set_header("CREATION_DATE", "2025-06-17T03:01:33.078351")

assert cdm0.header.get("CREATION_DATE") == "2025-06-17T03:01:33.078351"

cdm0.set_relative_metadata("RELATIVE_VELOCITY_R", 10)

assert cdm0.get_relative_metadata("RELATIVE_VELOCITY_R") == 10

cdm0.set_object("target", "REF_FRAME", "GCRF")
assert cdm0.get_object("target", "REF_FRAME") == "GCRF"

cdm0.set_object("target", "X", 10)
assert cdm0.get_object("target", "X") == 10

cdm0.set_object("target", "AREA_PC", 20)
assert cdm0.get_object("target", "AREA_PC") == 20

cdm0.set_object("target", "CR_R", 10)
assert cdm0.get_object("target", "CR_R") == 10

cdm0.set_object("chaser", "REF_FRAME", "GCRF")
assert cdm0.get_object("chaser", "REF_FRAME") == "GCRF"

cdm0.set_object("chaser", "X", 10)
assert cdm0.get_object("chaser", "X") == 10

cdm0.set_object("chaser", "AREA_PC", 20)
assert cdm0.get_object("chaser", "AREA_PC") == 20

cdm0.set_object("chaser", "CR_R", 10)
assert cdm0.get_object("chaser", "CR_R") == 10

In [79]:
state_values_target = [i * 10 for i in range(6)]

state_values_target = np.array(state_values_target)

state_matrix_target = state_values_target.reshape(2, 3)


state_values_chaser = [i * 5 for i in range(6)]

state_values_chaser = np.array(state_values_chaser)

state_matrix_chaser = state_values_chaser.reshape(2, 3)

cdm0.set_state("target", state_matrix_target)
cdm0.set_state("chaser", state_matrix_chaser)
assert np.array_equal(cdm0.get_state("target"), state_matrix_target)
assert np.array_equal(cdm0.get_state("chaser"), state_matrix_chaser)

2025-06-18 20:10:25.212 | WARNING  | kessler.io._CDM:_get_state_objects:281 - chaser has NaN values in its state.
2025-06-18 20:10:25.213 | WARNING  | kessler.io._CDM:_get_state_objects:281 - chaser has NaN values in its state.


In [80]:
covariance_values_target = [i * 10 for i in range(36)]

covariance_values_target = np.array(covariance_values_target)

covariance_matrix_target = covariance_values_target.reshape(6, 6)


covariance_values_chaser = [i * 5 for i in range(36)]

covariance_values_chaser = np.array(covariance_values_chaser)

covariance_matrix_chaser = covariance_values_chaser.reshape(6, 6)

In [ ]:
cdm0.set_covariance("target", covariance_matrix_target)
cdm0.set_covariance("chaser", covariance_matrix_chaser)

# TODO: this needs to be a covariance matrix
assert np.array_equal(cdm0.get_covariance("target"), covariance_matrix_target)
assert np.array_equal(cdm0.get_covariance("chaser"), covariance_matrix_chaser)

AssertionError: 

In [83]:
covariance_matrix_target

array([[  0,  10,  20,  30,  40,  50],
       [ 60,  70,  80,  90, 100, 110],
       [120, 130, 140, 150, 160, 170],
       [180, 190, 200, 210, 220, 230],
       [240, 250, 260, 270, 280, 290],
       [300, 310, 320, 330, 340, 350]])

In [82]:
cdm0.get_covariance("target")

array([[  0.,  60., 120., 180., 240., 300.],
       [ 60.,  70., 130., 190., 250., 310.],
       [120., 130., 140., 200., 260., 320.],
       [180., 190., 200., 210., 270., 330.],
       [240., 250., 260., 270., 280., 340.],
       [300., 310., 320., 330., 340., 350.]])

# Seperation

In [ ]:
columns_to_keep = list(cdm0.covariance_indices_dict.keys())
columns_to_keep = [f"t_{col}" for col in columns_to_keep]

ev0_filtered = ev0[columns_to_keep]
ev0_original_filtered = ev0_orginal[columns_to_keep]


In [ ]:
comparison = ev0_filtered.equals(ev0_original_filtered)
print("Are ev0_filtered and ev0_original_filtered equal?", comparison)

# Show differences if not equal
if not comparison:
    diff = ev0_filtered.compare(ev0_original_filtered)
    print("Differences between ev0_filtered and ev0_original_filtered:")
    print(diff)

In [ ]:
cdm0_original.get_object(0, "CR_R")

In [ ]:
import numpy as np

object_id = 0
covariance_original = np.zeros([6, 6])
covariance_original[0, 0] = cdm0_original.get_object(object_id, "CR_R")
covariance_original[1, 0] = cdm0_original.get_object(object_id, "CT_R")
covariance_original[1, 1] = cdm0_original.get_object(object_id, "CT_T")
covariance_original[2, 0] = cdm0_original.get_object(object_id, "CN_R")
covariance_original[2, 1] = cdm0_original.get_object(object_id, "CN_T")
covariance_original[2, 2] = cdm0_original.get_object(object_id, "CN_N")
covariance_original[3, 0] = cdm0_original.get_object(object_id, "CRDOT_R")
covariance_original[3, 1] = cdm0_original.get_object(object_id, "CRDOT_T")
covariance_original[3, 2] = cdm0_original.get_object(object_id, "CRDOT_N")
covariance_original[3, 3] = cdm0_original.get_object(object_id, "CRDOT_RDOT")
covariance_original[4, 0] = cdm0_original.get_object(object_id, "CTDOT_R")
covariance_original[4, 1] = cdm0_original.get_object(object_id, "CTDOT_T")
covariance_original[4, 2] = cdm0_original.get_object(object_id, "CTDOT_N")
covariance_original[4, 3] = cdm0_original.get_object(object_id, "CTDOT_RDOT")
covariance_original[4, 4] = cdm0_original.get_object(object_id, "CTDOT_TDOT")
covariance_original[5, 0] = cdm0_original.get_object(object_id, "CNDOT_R")
covariance_original[5, 1] = cdm0_original.get_object(object_id, "CNDOT_T")
covariance_original[5, 2] = cdm0_original.get_object(object_id, "CNDOT_N")
covariance_original[5, 3] = cdm0_original.get_object(object_id, "CNDOT_RDOT")
covariance_original[5, 4] = cdm0_original.get_object(object_id, "CNDOT_TDOT")
covariance_original[5, 5] = cdm0_original.get_object(object_id, "CNDOT_NDOT")
covariance_original

In [ ]:
covariance = np.zeros([6, 6])
for value in cdm0.keys_data_covariance_obligatory:
    i, j = cdm0.covariance_indices_dict.get(value)
    if i is not None and j is not None:
        covariance[i, j] = cdm0.get_object("target", value)

covariance

In [ ]:
are_equal = np.allclose(covariance_original, covariance)
print("Are covariance_original and covariance equal?", are_equal)

In [ ]:
np.allclose(cdm0.get_covariance("target"), cdm0_original.get_covariance(0))

In [ ]:
random_matrix = np.random.rand(6, 6)
cdm0.set_covariance("target", random_matrix)

In [ ]:
cdm0.get_covariance("target")

In [ ]:
df.loc[0, "c_cd_area_over_mass"]

In [ ]:
cdm0.get_state("target")

In [ ]:
cdm0_original.get_state(0)

In [ ]:
cdm0.keys_data_state_obligatory

In [ ]:
state_values = [0, 1, 2, 3, 4, 5]

for i, state_key in enumerate(cdm0.keys_data_state_obligatory):
    state_value = cdm0.get_object("target", state_key)
    print(f"State key: {state_key}, State value: {state_value}")
    cdm0.set_object("target", state_key, state_values[i])
    print(cdm0.get_object("target", state_key))
    print("-----------------------")

In [ ]:
for i, state_key in enumerate(cdm0.keys_data_state_obligatory):
    state_value = cdm0_original.get_object(0, state_key)
    print(f"State key: {state_key}, State value: {state_value}")
    cdm0_original.set_object(0, state_key, state_values[i])
    print(cdm0_original.get_object(0, state_key))
    print("-----------------------")

In [ ]:
cdm0_original.get_state(0)

In [ ]:
cdm0.get_state("target")

In [ ]:
state_values_target = [i * 10 for i in range(6)]

state_values_target = np.array(state_values_target)

state_matrix_target = state_values_target.reshape(2, 3)


state_values_chaser = [i * 5 for i in range(6)]

state_values_chaser = np.array(state_values_chaser)

state_matrix_chaser = state_values_chaser.reshape(2, 3)


In [ ]:
cdm0.set_state("target", state_matrix_target)
cdm0.set_state("chaser", state_matrix_chaser)

In [ ]:
cdm0.get_state("chaser")

In [ ]:
cdm0_original.set_state(0, state_matrix_target)
cdm0_original.set_state(1, state_matrix_chaser)

In [ ]:
cdm0_original.get_state(0)

In [ ]:
cdm0.relative_metadata

In [ ]:
for key, value in cdm0.relative_metadata.items():
    print(f"{key} : {cdm0_original.get_relative_metadata(key)}")

In [ ]:
np.allclose(cdm0.get_state_relative(), cdm0_original.get_state_relative())

In [ ]:
cdm0.validate()

In [ ]:
cdm0_original.validate()

In [ ]:
cdm0["MISS_DISTANCE"]

In [ ]:
cdm0 == events.events[0].cdms[0]